<a href="https://colab.research.google.com/github/Kaushgowda/infosys-franchise-ai/blob/main/Infosys_Milestone1_Single_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Infosys Springboard Internship 7.0 — Batch 4 | Milestone 1

This notebook combines the OTP page and login page into **one Streamlit application (`app.py`)**.

## Included features
- Signup and login
- Email-format validation
- Password complexity and confirm-password validation
- bcrypt password hashing
- Forgot password using a security question or email OTP
- OTP expires after 5 minutes; JWT-based login sessions
- Basic lockout after repeated failed login attempts
- Separate admin login and registered-user table (username/email only)
- User dashboard and logout

## Important setup
1. Run the install cell.
2. Run the `%%writefile app.py` cell to create the single application file.
3. In Colab, open the **Secrets (key icon)** and add `JWT_SECRET`, `NGROK_AUTHTOKEN`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD`, `ADMIN_USERNAME`, and `ADMIN_PASSWORD`.
4. Enable notebook access for each secret.
5. Run the launcher cell.

**Gmail:** `EMAIL_PASSWORD` must be a Gmail App Password, not your normal Gmail password. Keep all secrets private.

**Note:** This is a student project starter. Test all flows in Colab before submission.

In [1]:
!pip install -q streamlit pyngrok pyjwt bcrypt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 117.4 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import os
import re
import time
import sqlite3
import secrets
import smtplib
import datetime
from email.message import EmailMessage

import bcrypt
import jwt
import streamlit as st

# ------------------------------------------------------------
# Infosys Springboard Internship 7.0 - Batch 4 | Milestone 1
# Single-file Streamlit authentication application
# ------------------------------------------------------------

st.set_page_config(page_title="Infosys Portal", page_icon="🔐", layout="wide")

DB_PATH = "infosys_portal.db"
OTP_MINUTES = 5
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS = 300

JWT_SECRET = os.getenv("JWT_SECRET", "")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
ADMIN_USERNAME = os.getenv("ADMIN_USERNAME", "")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD", "")

# ---------- Styling ----------
st.markdown("""
<style>
.stApp { background: #f7fafc; }
.block-container { max-width: 1050px; padding-top: 2rem; }
h1, h2, h3 { color: #202747; }
div.stButton > button {
    border: 1px solid #202747; border-radius: 8px; font-weight: 600;
    min-height: 42px;
}
[data-testid="stSidebar"] { background: #e8f3f5; }
</style>
""", unsafe_allow_html=True)

# ---------- Database ----------
def connect_db():
    return sqlite3.connect(DB_PATH, check_same_thread=False)

def init_db():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE,
                email TEXT NOT NULL UNIQUE,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)
        # Migrate an existing database created by the earlier login-page code.
        columns = [row[1] for row in conn.execute("PRAGMA table_info(users)").fetchall()]
        if "created_at" not in columns:
            conn.execute("ALTER TABLE users ADD COLUMN created_at TEXT NOT NULL DEFAULT ''")

        conn.execute("""
            CREATE TABLE IF NOT EXISTS login_attempts (
                email TEXT PRIMARY KEY,
                attempts INTEGER NOT NULL DEFAULT 0,
                last_attempt REAL NOT NULL DEFAULT 0
            )
        """)

init_db()

# ---------- Helpers ----------
def hash_text(value):
    return bcrypt.hashpw(value.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")

def verify_hash(value, hashed_value):
    try:
        return bcrypt.checkpw(value.encode("utf-8"), hashed_value.encode("utf-8"))
    except (ValueError, TypeError, AttributeError):
        return False

def valid_email(email):
    # Requires at least 2 letters before @, at least 2 characters between @ and final dot,
    # and at least 2 letters after the final dot.
    pattern = r"^[A-Za-z0-9._%+-]{2,}@[A-Za-z0-9.-]{2,}\.[A-Za-z]{2,}$"
    return bool(re.fullmatch(pattern, email.strip()))

def valid_password(password):
    return (
        len(password) >= 8
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"\d", password)
        and re.search(r"[^A-Za-z0-9]", password)
    )

def password_rules_text():
    return "Use at least 8 characters, including uppercase, lowercase, a number, and a special character."

def make_token(email, role="user"):
    now = datetime.datetime.now(datetime.timezone.utc)
    payload = {
        "sub": email,
        "role": role,
        "iat": now,
        "exp": now + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def decode_token(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except (jwt.ExpiredSignatureError, jwt.InvalidTokenError, TypeError):
        return None

def send_otp_email(to_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Email secrets are missing. Add EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets."
    msg = EmailMessage()
    msg["Subject"] = "Infosys Portal - Password Reset OTP"
    msg["From"] = f"Infosys Portal <{EMAIL_ADDRESS}>"
    msg["To"] = to_email
    msg.set_content(
        f"Your Infosys Portal password reset OTP is: {otp}\n"
        f"This OTP expires in {OTP_MINUTES} minutes. If you did not request it, ignore this email."
    )
    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.starttls()
            server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
            server.send_message(msg)
        return True, "OTP sent successfully."
    except Exception as exc:
        return False, f"Could not send email. Check your Gmail App Password and secrets. Details: {exc}"

def user_exists(email):
    with connect_db() as conn:
        return conn.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone() is not None

def login_lock_status(email):
    with connect_db() as conn:
        row = conn.execute(
            "SELECT attempts, last_attempt FROM login_attempts WHERE email=?", (email,)
        ).fetchone()
    if not row:
        return False, 0
    attempts, last_attempt = row
    remaining = LOCKOUT_SECONDS - (time.time() - last_attempt)
    if attempts >= MAX_LOGIN_ATTEMPTS and remaining > 0:
        return True, int(remaining) + 1
    if attempts >= MAX_LOGIN_ATTEMPTS and remaining <= 0:
        with connect_db() as conn:
            conn.execute("DELETE FROM login_attempts WHERE email=?", (email,))
    return False, 0

def record_failed_login(email):
    with connect_db() as conn:
        row = conn.execute("SELECT attempts FROM login_attempts WHERE email=?", (email,)).fetchone()
        attempts = (row[0] if row else 0) + 1
        conn.execute(
            "INSERT OR REPLACE INTO login_attempts(email, attempts, last_attempt) VALUES (?, ?, ?)",
            (email, attempts, time.time())
        )

def clear_failed_logins(email):
    with connect_db() as conn:
        conn.execute("DELETE FROM login_attempts WHERE email=?", (email,))

def navigate(page):
    st.session_state.page = page
    st.rerun()

def logout():
    st.session_state.token = None
    st.session_state.page = "Login"
    st.session_state.reset_email = ""
    st.session_state.reset_method = ""
    st.session_state.otp_token = ""
    st.session_state.otp_verified = False
    st.rerun()

# ---------- Session state ----------
defaults = {
    "token": None,
    "page": "Login",
    "reset_email": "",
    "reset_method": "",
    "reset_question": "",
    "otp_token": "",
    "otp_verified": False,
}
for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value

st.title("🔐 Infosys Portal")
st.caption("Infosys Springboard Internship 7.0 — Batch 4 | Milestone 1")

# Configuration warning (do not show secret values)
missing = []
for name, value in [
    ("JWT_SECRET", JWT_SECRET),
    ("ADMIN_USERNAME", ADMIN_USERNAME),
    ("ADMIN_PASSWORD", ADMIN_PASSWORD),
]:
    if not value:
        missing.append(name)
if missing:
    st.warning("Add these missing items in Colab Secrets before using the app: " + ", ".join(missing))

# ---------- Admin authentication is separate from normal user records ----------
def admin_matches(username, password):
    return bool(ADMIN_USERNAME and ADMIN_PASSWORD and
                secrets.compare_digest(username, ADMIN_USERNAME) and
                secrets.compare_digest(password, ADMIN_PASSWORD))

# ---------- Auth pages ----------
payload = decode_token(st.session_state.token) if st.session_state.token else None
if st.session_state.token and not payload:
    st.session_state.token = None
    st.session_state.page = "Login"
    st.rerun()

if not payload:
    left, center, right = st.columns([1, 1.4, 1])
    with center:
        page = st.session_state.page

        if page == "Login":
            st.subheader("Login")
            login_type = st.radio("Login as", ["User", "Admin"], horizontal=True)
            if login_type == "User":
                email = st.text_input("Email address", key="login_email").strip().lower()
                password = st.text_input("Password", type="password", key="login_password")
                if st.button("Login", use_container_width=True):
                    if not email or not password:
                        st.error("Enter your email and password.")
                    elif not valid_email(email):
                        st.error("Enter a valid email address.")
                    else:
                        locked, seconds = login_lock_status(email)
                        if locked:
                            st.error(f"Too many failed attempts. Try again in about {seconds} seconds.")
                        else:
                            with connect_db() as conn:
                                row = conn.execute(
                                    "SELECT password_hash FROM users WHERE email=?", (email,)
                                ).fetchone()
                            if row and verify_hash(password, row[0]):
                                clear_failed_logins(email)
                                if not JWT_SECRET:
                                    st.error("JWT_SECRET is missing from Colab Secrets.")
                                else:
                                    st.session_state.token = make_token(email, "user")
                                    navigate("Dashboard")
                            else:
                                record_failed_login(email)
                                st.error("Invalid email or password.")
                c1, c2 = st.columns(2)
                if c1.button("Create account", use_container_width=True):
                    navigate("Signup")
                if c2.button("Forgot password?", use_container_width=True):
                    navigate("Forgot")
            else:
                admin_user = st.text_input("Admin username", key="admin_login_username")
                admin_pass = st.text_input("Admin password", type="password", key="admin_login_password")
                if st.button("Admin Login", use_container_width=True):
                    if not JWT_SECRET:
                        st.error("JWT_SECRET is missing from Colab Secrets.")
                    elif admin_matches(admin_user.strip(), admin_pass):
                        st.session_state.token = make_token("admin", "admin")
                        navigate("Admin Dashboard")
                    else:
                        st.error("Invalid admin username or password.")
                st.caption("Admin credentials are configured separately in Colab Secrets.")

        elif page == "Signup":
            st.subheader("Create Account")
            username = st.text_input("Full name / Username", key="signup_username").strip()
            email = st.text_input("Email address", key="signup_email").strip().lower()
            password = st.text_input("Password", type="password", key="signup_password")
            confirm = st.text_input("Confirm password", type="password", key="signup_confirm")
            questions = [
                "What was the name of your first school?",
                "What is your favourite city?",
                "What is your pet's name?"
            ]
            question = st.selectbox("Security question", questions)
            answer = st.text_input("Security answer", key="signup_answer").strip().lower()
            st.caption(password_rules_text())
            if st.button("Sign up", use_container_width=True):
                if not all([username, email, password, confirm, answer]):
                    st.error("Please complete all fields.")
                elif len(username) < 2:
                    st.error("Username must contain at least 2 characters.")
                elif not valid_email(email):
                    st.error("Enter a valid email. Use at least 2 characters before @, 2 between @ and the final dot, and 2 letters after the final dot.")
                elif not valid_password(password):
                    st.error("Password does not meet the required complexity. " + password_rules_text())
                elif password != confirm:
                    st.error("Password and confirm password do not match.")
                else:
                    try:
                        with connect_db() as conn:
                            conn.execute(
                                """INSERT INTO users(username,email,password_hash,security_question,
                                   security_answer_hash,created_at) VALUES(?,?,?,?,?,?)""",
                                (username, email, hash_text(password), question, hash_text(answer),
                                 datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                            )
                        st.success("Account created successfully. Please log in.")
                        st.session_state.page = "Login"
                        st.rerun()
                    except sqlite3.IntegrityError:
                        st.error("That username or email is already registered.")
            if st.button("Back to login"):
                navigate("Login")

        elif page == "Forgot":
            st.subheader("Forgot Password")
            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", key="forgot_email").strip().lower()
                if st.button("Continue", use_container_width=True):
                    if not valid_email(email):
                        st.error("Enter a valid email address.")
                    else:
                        with connect_db() as conn:
                            row = conn.execute(
                                "SELECT security_question FROM users WHERE email=?", (email,)
                            ).fetchone()
                        if not row:
                            st.error("No account was found with that email address.")
                        else:
                            st.session_state.reset_email = email
                            st.session_state.reset_question = row[0]
                            st.session_state.reset_method = ""
                            st.rerun()
            else:
                st.info(f"Account: {st.session_state.reset_email}")
                method = st.radio(
                    "Choose password reset method",
                    ["Security question", "Email OTP"],
                    horizontal=True,
                    key="forgot_method_radio"
                )
                st.session_state.reset_method = method
                if method == "Security question":
                    st.write("Security question:", st.session_state.reset_question)
                    answer = st.text_input("Your security answer", key="forgot_answer").strip().lower()
                    new_password = st.text_input("New password", type="password", key="forgot_new_password")
                    confirm_new = st.text_input("Confirm new password", type="password", key="forgot_confirm_password")
                    if st.button("Reset using security question", use_container_width=True):
                        if not valid_password(new_password):
                            st.error(password_rules_text())
                        elif new_password != confirm_new:
                            st.error("Passwords do not match.")
                        else:
                            with connect_db() as conn:
                                row = conn.execute(
                                    "SELECT security_answer_hash FROM users WHERE email=?",
                                    (st.session_state.reset_email,)
                                ).fetchone()
                            if row and verify_hash(answer, row[0]):
                                with connect_db() as conn:
                                    conn.execute(
                                        "UPDATE users SET password_hash=? WHERE email=?",
                                        (hash_text(new_password), st.session_state.reset_email)
                                    )
                                st.success("Password reset successfully. Please log in.")
                                st.session_state.reset_email = ""
                                st.session_state.reset_method = ""
                                st.session_state.otp_token = ""
                                st.session_state.otp_verified = False
                                st.session_state.page = "Login"
                                st.rerun()
                            else:
                                st.error("Incorrect security answer.")
                else:
                    if not st.session_state.otp_token:
                        if st.button("Send email OTP", use_container_width=True):
                            otp = f"{secrets.randbelow(900000) + 100000}"
                            sent, message = send_otp_email(st.session_state.reset_email, otp)
                            if sent:
                                now = datetime.datetime.now(datetime.timezone.utc)
                                otp_payload = {
                                    "sub": st.session_state.reset_email,
                                    "otp_hash": hash_text(otp),
                                    "purpose": "password_reset",
                                    "iat": now,
                                    "exp": now + datetime.timedelta(minutes=OTP_MINUTES)
                                }
                                st.session_state.otp_token = jwt.encode(otp_payload, JWT_SECRET, algorithm="HS256")
                                st.success(f"OTP sent. It expires in {OTP_MINUTES} minutes.")
                            else:
                                st.error(message)
                    else:
                        otp_input = st.text_input("Enter 6-digit OTP", max_chars=6, key="forgot_otp")
                        if not st.session_state.otp_verified and st.button("Verify OTP", use_container_width=True):
                            try:
                                otp_payload = jwt.decode(st.session_state.otp_token, JWT_SECRET, algorithms=["HS256"])
                                valid = (
                                    otp_payload.get("sub") == st.session_state.reset_email
                                    and otp_payload.get("purpose") == "password_reset"
                                    and verify_hash(otp_input, otp_payload.get("otp_hash", ""))
                                )
                                if valid:
                                    st.session_state.otp_verified = True
                                    st.success("OTP verified.")
                                    st.rerun()
                                else:
                                    st.error("Incorrect OTP.")
                            except jwt.ExpiredSignatureError:
                                st.session_state.otp_token = ""
                                st.error("OTP expired. Please send a new OTP.")
                            except jwt.InvalidTokenError:
                                st.session_state.otp_token = ""
                                st.error("Invalid OTP session. Please send a new OTP.")
                        if st.session_state.otp_verified:
                            new_password = st.text_input("New password", type="password", key="otp_new_password")
                            confirm_new = st.text_input("Confirm new password", type="password", key="otp_confirm_password")
                            if st.button("Update password", use_container_width=True):
                                if not valid_password(new_password):
                                    st.error(password_rules_text())
                                elif new_password != confirm_new:
                                    st.error("Passwords do not match.")
                                else:
                                    with connect_db() as conn:
                                        conn.execute(
                                            "UPDATE users SET password_hash=? WHERE email=?",
                                            (hash_text(new_password), st.session_state.reset_email)
                                        )
                                    st.success("Password reset successfully. Please log in.")
                                    st.session_state.reset_email = ""
                                    st.session_state.reset_method = ""
                                    st.session_state.otp_token = ""
                                    st.session_state.otp_verified = False
                                    st.session_state.page = "Login"
                                    st.rerun()
                if st.button("Cancel and return to login"):
                    st.session_state.reset_email = ""
                    st.session_state.reset_method = ""
                    st.session_state.otp_token = ""
                    st.session_state.otp_verified = False
                    navigate("Login")

# ---------- Dashboards ----------
else:
    role = payload.get("role")
    if role == "admin":
        st.sidebar.title("Admin Panel")
        st.sidebar.write("Signed in as administrator")
        if st.sidebar.button("Logout", use_container_width=True):
            logout()
        st.header("🛡️ Admin Dashboard")
        st.write("Registered users")
        with connect_db() as conn:
            rows = conn.execute(
                "SELECT username, email, created_at FROM users ORDER BY created_at DESC"
            ).fetchall()
        st.metric("Total registered users", len(rows))
        if rows:
            st.dataframe(
                [{"Username": row[0], "Email": row[1], "Registered At": row[2]} for row in rows],
                use_container_width=True,
                hide_index=True
            )
        else:
            st.info("No registered users yet.")
        st.caption("Passwords and password hashes are never displayed.")
    else:
        email = payload.get("sub")
        with connect_db() as conn:
            row = conn.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()
        if not row:
            st.error("User account was not found. Please log in again.")
            logout()
        username = row[0]
        st.sidebar.title("User Menu")
        st.sidebar.write(f"Welcome, {username}")
        if st.sidebar.button("Logout", use_container_width=True):
            logout()
        st.header(f"Welcome, {username}!")
        st.success("You are successfully logged in.")
        st.subheader("User Dashboard")
        st.write("Your account session is protected with a JWT token.")
        st.info("Use the sidebar Logout button to end your session.")


Writing app.py


In [1]:


import os
import time
import subprocess
from google.colab import userdata
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError

required_secrets = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS",
                    "EMAIL_PASSWORD", "ADMIN_USERNAME", "ADMIN_PASSWORD"]
missing = []
for key in required_secrets:
    try:
        value = userdata.get(key)
        if not value:
            missing.append(key)
        else:
            os.environ[key] = value
    except Exception:
        missing.append(key)

if missing:
    raise ValueError("Add these missing secrets in Colab Secrets and enable notebook access: " + ", ".join(missing))

try:
    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
except PyngrokNgrokError as e:
    if "Authtoken already set" not in str(e):
        raise

# Stop an old Streamlit server if this notebook was run earlier.
subprocess.run("pkill -f 'streamlit run app.py'", shell=True, check=False)
time.sleep(2)

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)

try:
    public_url = ngrok.connect(8501, "http").public_url
    print("Infosys Portal is running.")
    print("Open this URL:", public_url)
    print("Keep this cell running while you use the app.")
except Exception:
    process.terminate()
    raise

Infosys Portal is running.
Open this URL: https://footpad-genre-singer.ngrok-free.dev
Keep this cell running while you use the app.
